# Linguagens de Programação para Engenharia de Dados
## Encontro 6 — Notebook Prático

**Instituição:** Universidade de Fortaleza (UNIFOR)
**Curso:** Pós-Graduação — Especialização em Engenharia de Dados
**Professor:** Cassio Pinheiro — [LinkedIn](https://www.linkedin.com/in/cassio-pinheiro-9baa7939/)
**Data:** 11/07/2026 · 19:00 às 22:30

---

### Como usar este notebook

1. Execute as células **na ordem**, de cima para baixo (`Shift + Enter`).
2. Cada seção corresponde a uma seção do documento `.md` do encontro.
3. O **núcleo** roda com **biblioteca padrão do Python + pandas + pyarrow** (Parquet). As seções 3 e 4 são **ilustrativas**: mostram o mesmo DAG em Airflow, Prefect e Dagster — **sem instalar** essas bibliotecas.
4. **Mexa no código.** Quebre uma tarefa de propósito e veja o retry recuperar.

> **Projeto fio condutor (E1→E6):** o mesmo `vendas.csv` sintético e sujo do Encontro 1, agora levado de ponta a ponta por um **mini-orquestrador de DAG escrito em Python puro** — extrair → transformar → validar → carregar (Parquet) → sumarizar. É o pipeline do curso inteiro, agora autônomo.

## Setup

Biblioteca padrão (`csv`, `io`, `time`, `datetime`, `functools`) + **pandas** (transformação) + **pyarrow** (Parquet). Nada de Airflow/Prefect/Dagster aqui — esses aparecem só como **código ilustrativo** mais à frente.

In [1]:
import csv
import io
import time
from datetime import datetime
import pandas as pd

print("Setup OK — pandas", pd.__version__)
print("Núcleo executável: Python puro + pandas + pyarrow.")

Setup OK — pandas 2.3.3
Núcleo executável: Python puro + pandas + pyarrow.


---

# 1 — De script a sistema

> 📖 **No `.md`:** seção *1 — De script a pipeline orquestrado*.

Depois de cinco encontros, o pipeline do curso existe — mas como **funções soltas** que um humano chama em ordem. Vamos primeiro recriar o `vendas.csv` sujo do E1 e escrever as etapas como funções. O problema: **nada garante a ordem nem trata a falha**. É exatamente o que a orquestração resolve.

In [2]:
# O MESMO csv sujo do Encontro 1 (datas em 3 formatos, valor vazio/negativo,
# duplicata, data vazia, espacos sobrando). Fio condutor E1 -> E6.
CSV_BRUTO = """id,data,cliente,categoria,valor
1,2026-06-01,Ana Souza,eletronicos,1200.00
2,01/06/2026,  Bruno Lima ,alimentacao,85.50
3,2026-06-02,Carla Dias,eletronicos,
4,2026-06-02,Diego Reis,transporte,-30.00
5,2026/06/03,Ana Souza,alimentacao,42.90
5,2026/06/03,Ana Souza,alimentacao,42.90
6,2026-06-03,Elena Cruz,vestuario,260.00
7,,Felipe Aragao,servicos,99.90
"""

print("Dado bruto (fonte legada):")
print(CSV_BRUTO)
print("Como FUNCOES soltas, a ordem certa so existe na cabeca de quem executa.")
print("Se 'carregar' rodar antes de 'transformar', le um dado que ainda nao existe.")

Dado bruto (fonte legada):
id,data,cliente,categoria,valor
1,2026-06-01,Ana Souza,eletronicos,1200.00
2,01/06/2026,  Bruno Lima ,alimentacao,85.50
3,2026-06-02,Carla Dias,eletronicos,
4,2026-06-02,Diego Reis,transporte,-30.00
5,2026/06/03,Ana Souza,alimentacao,42.90
5,2026/06/03,Ana Souza,alimentacao,42.90
6,2026-06-03,Elena Cruz,vestuario,260.00
7,,Felipe Aragao,servicos,99.90

Como FUNCOES soltas, a ordem certa so existe na cabeca de quem executa.
Se 'carregar' rodar antes de 'transformar', le um dado que ainda nao existe.


---

# 2 — Um mini-orquestrador de DAG (Python puro)

> 📖 **No `.md`:** seção *2 — DAGs*.

Vamos construir, do zero, o que um orquestrador faz por baixo:

1. **declarar tarefas** como funções, com **dependências** num dicionário;
2. resolver a **ordem topológica** (toda tarefa depois de suas dependências);
3. executar com **retries** (contador), **logging estruturado** (timestamp + status) e **idempotência**.

Primeiro, o motor do orquestrador.

In [3]:
# ---------- LOGGING ESTRUTURADO ----------
LOG = []   # registro de tudo que o orquestrador fez (rastreabilidade do E1)

def log(tarefa, status, detalhe=""):
    """Registra um evento com timestamp, tarefa e status (rastreabilidade)."""
    ts = datetime.now().strftime("%H:%M:%S")
    LOG.append({"ts": ts, "tarefa": tarefa, "status": status, "detalhe": detalhe})
    print(f"[{ts}] {tarefa:12s} {status:9s} {detalhe}")

# ---------- ORDEM TOPOLOGICA ----------
def ordem_topologica(deps):
    """Recebe {tarefa: [dependencias]} e devolve uma ordem de execucao valida.
    Detecta ciclos (o 'A' de aciclico do DAG)."""
    visitado, na_pilha, ordem = set(), set(), []
    def visita(t):
        if t in na_pilha:
            raise ValueError(f"CICLO detectado envolvendo '{t}' — DAG invalido!")
        if t in visitado:
            return
        na_pilha.add(t)
        for d in deps.get(t, []):
            visita(d)
        na_pilha.discard(t); visitado.add(t); ordem.append(t)
    for t in deps:
        visita(t)
    return ordem

# Teste rapido do algoritmo com um grafo simples:
exemplo = {"sumarizar": ["carregar"], "carregar": ["validar"],
           "validar": ["transformar"], "transformar": ["extrair"], "extrair": []}
print("Ordem topologica:", ordem_topologica(exemplo))

Ordem topologica: ['extrair', 'transformar', 'validar', 'carregar', 'sumarizar']


O motor de execução: percorre a ordem topológica, executa cada tarefa **com retries** e registra tudo. O estado do pipeline trafega num dicionário `contexto` — cada tarefa lê o que precisa e grava o que produz.

In [4]:
# ---------- EXECUTOR COM RETRIES ----------
def executar_dag(tarefas, deps, contexto, max_retries=2, espera=0.0):
    """Executa o DAG na ordem topologica. Cada tarefa eh uma funcao
    contexto -> contexto. Retries reexecutam tarefas que levantam excecao."""
    ordem = ordem_topologica(deps)
    log("orquestrador", "INICIO", f"ordem: {' -> '.join(ordem)}")
    for nome in ordem:
        fn = tarefas[nome]
        tentativa = 0
        while True:
            tentativa += 1
            try:
                contexto = fn(contexto)
                marca = "OK" if tentativa == 1 else f"OK(ret={tentativa-1})"
                log(nome, marca)
                break
            except Exception as e:
                if tentativa <= max_retries:
                    log(nome, "RETRY", f"tentativa {tentativa} falhou: {e}")
                    time.sleep(espera)
                    continue
                log(nome, "FALHOU", f"esgotou {max_retries} retries: {e}")
                raise
    log("orquestrador", "FIM", "pipeline concluido")
    return contexto

print("Executor pronto: ordem topologica + retries + logging estruturado.")

Executor pronto: ordem topologica + retries + logging estruturado.


Agora as **cinco tarefas do pipeline do curso**. Cada uma é idempotente: recebe o `contexto`, produz seu resultado e o devolve — rodar de novo dá o mesmo resultado.

In [5]:
# ---------- TAREFA 1: EXTRAIR ----------
def extrair(ctx):
    linhas = list(csv.DictReader(io.StringIO(CSV_BRUTO)))
    ctx["brutas"] = linhas
    ctx["log_contagem"] = {"lidas": len(linhas)}
    return ctx

# ---------- TAREFA 2: TRANSFORMAR (limpeza com pandas) ----------
def _padroniza_data(txt):
    txt = (txt or "").strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d"):
        try:
            return datetime.strptime(txt, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

def transformar(ctx):
    df = pd.DataFrame(ctx["brutas"])
    df["cliente"]   = df["cliente"].str.strip()
    df["categoria"] = df["categoria"].str.strip()
    df["data"]      = df["data"].apply(_padroniza_data)
    # valor: vazio -> NaN; converte para numero
    df["valor"] = pd.to_numeric(df["valor"].replace("", None), errors="coerce")
    ctx["transformadas"] = df
    return ctx

# ---------- TAREFA 3: VALIDAR (qualidade + quarentena, heranca do E5) ----------
def validar(ctx):
    df = ctx["transformadas"].copy()
    motivo = pd.Series([""] * len(df), index=df.index)
    motivo[df["valor"].isna()]       = "valor_ausente"
    motivo[(motivo == "") & (df["valor"] < 0)] = "valor_negativo"
    motivo[(motivo == "") & (df["data"].isna())] = "data_invalida"
    dup = df.duplicated(subset=["id", "data", "valor"], keep="first")
    motivo[(motivo == "") & dup] = "duplicata"
    ctx["limpas"]     = df[motivo == ""].copy()
    ctx["quarentena"] = df[motivo != ""].assign(motivo_quarentena=motivo[motivo != ""])
    ctx["log_contagem"]["aprovadas"]  = len(ctx["limpas"])
    ctx["log_contagem"]["quarentena"] = len(ctx["quarentena"])
    return ctx

# ---------- TAREFA 4: CARREGAR (Parquet, heranca do E4) ----------
def carregar(ctx):
    df = ctx["limpas"].copy()
    df["valor"] = df["valor"].astype(float)
    df["total_com_imposto"] = (df["valor"] * 1.18).round(2)
    df.to_parquet("/tmp/vendas_tratadas.parquet", index=False)  # idempotente: sobrescreve
    ctx["carregadas"] = df
    return ctx

# ---------- TAREFA 5: SUMARIZAR (faturamento por categoria) ----------
def sumarizar(ctx):
    df = pd.read_parquet("/tmp/vendas_tratadas.parquet")
    ctx["faturamento"] = (df.groupby("categoria")["valor"].sum()
                            .sort_values(ascending=False))
    return ctx

print("5 tarefas definidas: extrair, transformar, validar, carregar, sumarizar.")

5 tarefas definidas: extrair, transformar, validar, carregar, sumarizar.


As tarefas e suas **dependências declaradas**. Note: descrevemos *quem depende de quem*, não a ordem — o orquestrador deriva a ordem sozinho.

In [6]:
TAREFAS = {
    "extrair":     extrair,
    "transformar": transformar,
    "validar":     validar,
    "carregar":    carregar,
    "sumarizar":   sumarizar,
}
DEPS = {
    "extrair":     [],
    "transformar": ["extrair"],
    "validar":     ["transformar"],
    "carregar":    ["validar"],
    "sumarizar":   ["carregar"],
}

print("DAG do pipeline do curso:")
print("  " + " -> ".join(ordem_topologica(DEPS)))

DAG do pipeline do curso:
  extrair -> transformar -> validar -> carregar -> sumarizar


**Execução 1 — caminho feliz.** O orquestrador resolve a ordem e roda tudo, registrando cada passo.

In [7]:
LOG.clear()
ctx = executar_dag(TAREFAS, DEPS, contexto={})

print("\n--- Resultado: faturamento por categoria (dado tratado) ---")
for cat, total in ctx["faturamento"].items():
    print(f"  {cat:14s} R$ {total:>9.2f}")

print(f"\nContagens: {ctx['log_contagem']}")
print(f"Quarentenadas: {len(ctx['quarentena'])} linhas (refugo do E5)")

[19:57:37] orquestrador INICIO    ordem: extrair -> transformar -> validar -> carregar -> sumarizar
[19:57:37] extrair      OK        
[19:57:37] transformar  OK        
[19:57:37] validar      OK        
[19:57:37] carregar     OK        
[19:57:37] sumarizar    OK        
[19:57:37] orquestrador FIM       pipeline concluido

--- Resultado: faturamento por categoria (dado tratado) ---
  eletronicos    R$   1200.00
  vestuario      R$    260.00
  alimentacao    R$    128.40

Contagens: {'lidas': 8, 'aprovadas': 4, 'quarentena': 4}
Quarentenadas: 4 linhas (refugo do E5)


---

# 3 — O mesmo DAG em Apache Airflow (ilustrativo)

> 📖 **No `.md`:** seção *3 — Apache Airflow*.

O mini-orquestrador da seção 2 fez **à mão** o que o Airflow faz como produto: ordem topológica, retries, logging. Em produção, escreveríamos o DAG como código Python. **A célula abaixo apenas EXIBE o código** — não instalamos o Airflow.

In [8]:
CODIGO_AIRFLOW = '''
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

with DAG(
    dag_id="pipeline_vendas",
    schedule="0 6 * * *",          # cron: todo dia as 6h
    start_date=datetime(2026, 7, 1),
    catchup=False,
    default_args={"retries": 2},   # mesma ideia do nosso executor
) as dag:

    t_extrair     = PythonOperator(task_id="extrair",     python_callable=extrair)
    t_transformar = PythonOperator(task_id="transformar", python_callable=transformar)
    t_validar     = PythonOperator(task_id="validar",     python_callable=validar)
    t_carregar    = PythonOperator(task_id="carregar",    python_callable=carregar)
    t_sumarizar   = PythonOperator(task_id="sumarizar",   python_callable=sumarizar)

    # ">>" declara a aresta do DAG (a nossa DEPS, em forma legivel):
    t_extrair >> t_transformar >> t_validar >> t_carregar >> t_sumarizar
'''
print(CODIGO_AIRFLOW)
print(">> Repare: 'DEPS' (nosso dict) vira a linha com '>>'.")
print(">> 'default_args.retries' vira o nosso 'max_retries'.")
print(">> Airflow eh centrado em TAREFAS: sabe que rodou, nao no dado produzido.")


from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

with DAG(
    dag_id="pipeline_vendas",
    schedule="0 6 * * *",          # cron: todo dia as 6h
    start_date=datetime(2026, 7, 1),
    catchup=False,
    default_args={"retries": 2},   # mesma ideia do nosso executor
) as dag:

    t_extrair     = PythonOperator(task_id="extrair",     python_callable=extrair)
    t_transformar = PythonOperator(task_id="transformar", python_callable=transformar)
    t_validar     = PythonOperator(task_id="validar",     python_callable=validar)
    t_carregar    = PythonOperator(task_id="carregar",    python_callable=carregar)
    t_sumarizar   = PythonOperator(task_id="sumarizar",   python_callable=sumarizar)

    # ">>" declara a aresta do DAG (a nossa DEPS, em forma legivel):
    t_extrair >> t_transformar >> t_validar >> t_carregar >> t_sumarizar

>> Repare: 'DEPS' (nosso dict) vira a linha com '>>'.
>> 'default_args.retries' vira o nosso '

---

# 4 — O mesmo pipeline em Prefect e Dagster (ilustrativo)

> 📖 **No `.md`:** seção *4 — Orquestração asset-based*.

A tendência atual é **asset-based**: o foco passa da tarefa para o **dado produzido**. Veja o mesmo pipeline em **Prefect** (flows/tasks dinâmicos) e **Dagster** (assets). De novo: **só exibimos o código**, sem instalar nada.

In [9]:
CODIGO_PREFECT = '''
from prefect import flow, task

@task(retries=2)                       # retries por tarefa, como no nosso executor
def extrair(): ...
@task
def transformar(brutas): ...
@task
def validar(transformadas): ...
@task
def carregar(limpas): ...
@task
def sumarizar(): ...

@flow                                  # o flow orquestra; deps nascem do USO do resultado
def pipeline_vendas():
    brutas        = extrair()
    transformadas = transformar(brutas)
    limpas        = validar(transformadas)
    carregar(limpas)
    return sumarizar()
'''
print(CODIGO_PREFECT)
print(">> Prefect: chama tasks como funcoes normais; a dependencia vem do dado passado.")
print(">> Diferencial: dinamismo — o grafo pode ser construido em tempo de execucao.")


from prefect import flow, task

@task(retries=2)                       # retries por tarefa, como no nosso executor
def extrair(): ...
@task
def transformar(brutas): ...
@task
def validar(transformadas): ...
@task
def carregar(limpas): ...
@task
def sumarizar(): ...

@flow                                  # o flow orquestra; deps nascem do USO do resultado
def pipeline_vendas():
    brutas        = extrair()
    transformadas = transformar(brutas)
    limpas        = validar(transformadas)
    carregar(limpas)
    return sumarizar()

>> Prefect: chama tasks como funcoes normais; a dependencia vem do dado passado.
>> Diferencial: dinamismo — o grafo pode ser construido em tempo de execucao.


In [10]:
CODIGO_DAGSTER = '''
from dagster import asset

@asset
def vendas_brutas():                       # o ATIVO, nao a tarefa
    return ler_csv("vendas.csv")

@asset
def vendas_limpas(vendas_brutas):          # depende do ativo acima (a aresta eh o argumento)
    return transformar(vendas_brutas)

@asset
def vendas_parquet(vendas_limpas):
    return carregar_parquet(vendas_limpas)

@asset
def faturamento_por_categoria(vendas_parquet):
    return sumarizar(vendas_parquet)
'''
print(CODIGO_DAGSTER)
print(">> Dagster: voce declara os ATIVOS (dados) e suas dependencias; o DAG eh derivado.")
print(">> A linhagem (lineage) vem de graca — materializa 'o dado como produto' do E1.")
print(">> Quando preferir Airflow: time ja o domina, jobs heterogeneos nao-data-asset.")


from dagster import asset

@asset
def vendas_brutas():                       # o ATIVO, nao a tarefa
    return ler_csv("vendas.csv")

@asset
def vendas_limpas(vendas_brutas):          # depende do ativo acima (a aresta eh o argumento)
    return transformar(vendas_brutas)

@asset
def vendas_parquet(vendas_limpas):
    return carregar_parquet(vendas_limpas)

@asset
def faturamento_por_categoria(vendas_parquet):
    return sumarizar(vendas_parquet)

>> Dagster: voce declara os ATIVOS (dados) e suas dependencias; o DAG eh derivado.
>> A linhagem (lineage) vem de graca — materializa 'o dado como produto' do E1.
>> Quando preferir Airflow: time ja o domina, jobs heterogeneos nao-data-asset.


---

# 5 — Produção: retries recuperando uma falha (executável)

> 📖 **No `.md`:** seção *5 — Produção*.

Em produção, falhas transitórias acontecem (rede instável, lock momentâneo). O retry existe para isso. Vamos **simular** uma tarefa de carga que falha nas 2 primeiras tentativas e **recupera na 3ª** — sem que a falha derrube o pipeline. Tudo fica no log estruturado.

In [11]:
# Tarefa de carga INSTAVEL: falha nas 2 primeiras chamadas, sucede na 3a.
# (A falha eh SIMULADA e tratada pelo retry — nao se propaga.)
_tentativas_carga = {"n": 0}

def carregar_instavel(ctx):
    _tentativas_carga["n"] += 1
    if _tentativas_carga["n"] < 3:
        raise ConnectionError("falha transitoria de rede ao gravar Parquet")
    return carregar(ctx)   # na 3a tentativa, faz a carga de verdade

# Monta um DAG igual, trocando 'carregar' pela versao instavel.
TAREFAS_PROD = dict(TAREFAS, carregar=carregar_instavel)

LOG.clear()
_tentativas_carga["n"] = 0
ctx2 = executar_dag(TAREFAS_PROD, DEPS, contexto={}, max_retries=2, espera=0.0)

print("\n>> A tarefa 'carregar' FALHOU 2x e RECUPEROU na 3a — pipeline concluido.")
print(">> Em producao, se esgotasse os retries, viraria ALERTA (email/Slack/PagerDuty).")

[19:57:37] orquestrador INICIO    ordem: extrair -> transformar -> validar -> carregar -> sumarizar
[19:57:37] extrair      OK        
[19:57:37] transformar  OK        
[19:57:37] validar      OK        
[19:57:37] carregar     RETRY     tentativa 1 falhou: falha transitoria de rede ao gravar Parquet
[19:57:37] carregar     RETRY     tentativa 2 falhou: falha transitoria de rede ao gravar Parquet
[19:57:37] carregar     OK(ret=2) 
[19:57:37] sumarizar    OK        
[19:57:37] orquestrador FIM       pipeline concluido

>> A tarefa 'carregar' FALHOU 2x e RECUPEROU na 3a — pipeline concluido.
>> Em producao, se esgotasse os retries, viraria ALERTA (email/Slack/PagerDuty).


O **log estruturado** é a observabilidade do pipeline: um registro auditável de cada evento, com timestamp e status. É a rastreabilidade do E1 virando ferramenta de produção.

In [12]:
log_df = pd.DataFrame(LOG)
print("Log estruturado da execucao (observabilidade):")
print(log_df.to_string(index=False))

print("\nResumo por status:")
print(log_df["status"].value_counts().to_string())

Log estruturado da execucao (observabilidade):
      ts       tarefa    status                                                           detalhe
19:57:37 orquestrador    INICIO ordem: extrair -> transformar -> validar -> carregar -> sumarizar
19:57:37      extrair        OK                                                                  
19:57:37  transformar        OK                                                                  
19:57:37      validar        OK                                                                  
19:57:37     carregar     RETRY   tentativa 1 falhou: falha transitoria de rede ao gravar Parquet
19:57:37     carregar     RETRY   tentativa 2 falhou: falha transitoria de rede ao gravar Parquet
19:57:37     carregar OK(ret=2)                                                                  
19:57:37    sumarizar        OK                                                                  
19:57:37 orquestrador       FIM                                        

---

# Encerramento do curso — o pipeline E1→E6 como um sistema único

> 📖 **No `.md`:** seção *6 — Fechamento da disciplina*.

Uma última execução, limpa, do pipeline completo — e o mapa de como cada encontro contribuiu para ele.

In [13]:
LOG.clear()
ctx_final = executar_dag(TAREFAS, DEPS, contexto={})

print("\n=== O pipeline do curso, orquestrado de ponta a ponta ===")
print(f"Lidas:      {ctx_final['log_contagem']['lidas']}")
print(f"Aprovadas:  {ctx_final['log_contagem']['aprovadas']}")
print(f"Quarentena: {ctx_final['log_contagem']['quarentena']}")
print("\nFaturamento por categoria (dado confiavel, em Parquet):")
for cat, total in ctx_final["faturamento"].items():
    print(f"  {cat:14s} R$ {total:>9.2f}")

[19:57:37] orquestrador INICIO    ordem: extrair -> transformar -> validar -> carregar -> sumarizar
[19:57:37] extrair      OK        
[19:57:37] transformar  OK        
[19:57:37] validar      OK        
[19:57:37] carregar     OK        
[19:57:37] sumarizar    OK        
[19:57:37] orquestrador FIM       pipeline concluido

=== O pipeline do curso, orquestrado de ponta a ponta ===
Lidas:      8
Aprovadas:  4
Quarentena: 4

Faturamento por categoria (dado confiavel, em Parquet):
  eletronicos    R$   1200.00
  vestuario      R$    260.00
  alimentacao    R$    128.40


In [14]:
mapa = [
    ("E1 Fundamentos",   "extrair->transformar->carregar; idempotencia, rastreabilidade"),
    ("E2 pandas",        "laços viraram operacoes de tabela (vide 'transformar')"),
    ("E3 SQL/transform", "pensar em conjuntos; mesma pergunta em SQL e em Python"),
    ("E4 Escala",        "carga em Parquet, particionamento, volume"),
    ("E5 Qualidade",     "validar/quarentena: o refugo nao polui o resultado"),
    ("E6 Orquestracao",  "DAG: ordem topologica + retries + logging + idempotencia"),
]
print("A DISCIPLINA EM UM MAPA — os 6 encontros, um so pipeline\n" + "="*64)
for enc, contrib in mapa:
    print(f"{enc:18s} -> {contrib}")
print("\nOs principios nao mudaram; ganharam um corpo que se executa sozinho.")

A DISCIPLINA EM UM MAPA — os 6 encontros, um so pipeline
E1 Fundamentos     -> extrair->transformar->carregar; idempotencia, rastreabilidade
E2 pandas          -> laços viraram operacoes de tabela (vide 'transformar')
E3 SQL/transform   -> pensar em conjuntos; mesma pergunta em SQL e em Python
E4 Escala          -> carga em Parquet, particionamento, volume
E5 Qualidade       -> validar/quarentena: o refugo nao polui o resultado
E6 Orquestracao    -> DAG: ordem topologica + retries + logging + idempotencia

Os principios nao mudaram; ganharam um corpo que se executa sozinho.


---

## 🧪 Exercícios do Encontro 6

Resolva no próprio notebook, criando novas células abaixo:

1. **Nova tarefa no DAG.** Adicione uma tarefa `notificar` que depende de `sumarizar` e imprime "pipeline OK" com a contagem de aprovadas. Atualize `TAREFAS` e `DEPS` e rode — confirme que a ordem topológica a coloca **por último**.
2. **Detecção de ciclo.** Crie um `DEPS` propositalmente cíclico (ex.: `extrair` depende de `sumarizar`) e chame `ordem_topologica`. Confirme que ele levanta o erro de ciclo — o "A" de **acíclico** protegendo o pipeline.
3. **Retries no limite.** Faça `carregar_instavel` falhar **4 vezes** (mude o `< 3` para `< 5`) e rode com `max_retries=2`. Observe o pipeline **falhar de forma clara** (em produção, isso seria um alerta) — e depois conserte aumentando `max_retries`.
4. **Reflexão (escrever, não codar).** Pegue um processo de dados do seu trabalho e desenhe-o como DAG: liste as tarefas, as dependências, o agendamento (cron) e o que deveria gerar um alerta. É o seu primeiro projeto de portfólio.

> Dica: o melhor exercício é **quebrar o DAG de propósito** (ciclo, falha sem retry) e consertá-lo. Em orquestração, entender a falha é metade do trabalho.

---

## Encerramento do curso

Este notebook acompanha o documento `.md` (teoria detalhada) e os slides `.pptx` (aula expositiva) do **Encontro 6 — o último da disciplina**.

Hoje você: construiu um **mini-orquestrador de DAG em Python puro** (ordem topológica, retries, logging estruturado, idempotência); orquestrou o **pipeline completo do curso** (extrair → transformar → validar/quarentena → carregar Parquet → sumarizar); viu o **mesmo DAG** em Airflow, Prefect e Dagster; e simulou uma **falha recuperada por retry** em produção.

Mais do que isso: você fechou o ciclo. O `vendas.csv` sujo do Encontro 1 chegou hoje ao fim da jornada como um **sistema que roda sozinho**. Os princípios plantados na primeira aula — idempotência, tratamento de erro, rastreabilidade, observabilidade — não eram teoria: eram o projeto deste pipeline desde o começo.

> **Um script roda uma vez; um pipeline roda sempre, sozinho e confiável.** Construir esse "sempre" é a Engenharia de Dados. Bons pipelines — e até a próxima.

**Bibliografia:** Reis & Housley, *Fundamentos de Engenharia de Dados* (2023) · McKinney, *Python para Análise de Dados* (2022) · Kleppmann, *Designing Data-Intensive Applications* (2017).